In [ ]:
# 套件
import cv2
import numpy as np
import mediapipe as mp

from mediapipe import solutions
from mediapipe.framework.formats import landmark_pb2
from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision.pose_landmarker import (
    PoseLandmarker,
    PoseLandmarkerOptions,
    PoseLandmarkerResult,
)
from mediapipe.tasks.python.vision.core.vision_task_running_mode import (
    VisionTaskRunningMode,
)

In [ ]:
# 常數
MODEL_PATH = "./models/BlazePose/pose_landmarker_full.task"
IMG_PATH = "./images/pexels-photo-4384679.jpeg"

In [ ]:
def draw_landmarks_on_image(rgb_image, detection_result: PoseLandmarkerResult):
    pose_landmark_list = detection_result.pose_landmarks
    annotated_image = np.copy(rgb_image)

    for idx in range(len(pose_landmark_list)):
        pose_landmarks = pose_landmark_list[idx]

        pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
        pose_landmarks_proto.landmark.extend(
            [
                landmark_pb2.NormalizedLandmark(
                    x=landmark.x, y=landmark.y, z=landmark.z
                )
                for landmark in pose_landmarks
            ]
        )
        solutions.drawing_utils.draw_landmarks(
            annotated_image,
            pose_landmarks_proto,
            solutions.pose.POSE_CONNECTIONS,
            solutions.drawing_styles.get_default_pose_landmarks_style(),
        )

        return annotated_image

In [ ]:
# 模型設定
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionTaskRunningMode.IMAGE,
)

# 計算模型結果
with PoseLandmarker.create_from_options(options) as landmarker:
    cv_mat = cv2.imread(IMG_PATH)
    
    h, w, c = cv_mat.shape
    cv_mat = cv2.resize(cv_mat, (w // 5, h // 5))
    
    img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv_mat)
    result = landmarker.detect(img)

    annotated_image = draw_landmarks_on_image(cv_mat, result)
    cv2.imshow("result", annotated_image)
    cv2.waitKey(0)

cv2.destroyAllWindows()

In [ ]:
# cap = cv2.VideoCapture(0)
# while cap.isOpened():
#     ret, frame = cap.read()


#     cv2.imshow("webcam", frame)

#     if not ret:
#         print("Can't receive frame (stream end?). Exiting ...")
#     if cv2.waitKey(1) == ord("q"):
#         break

# cap.release()
# cv2.destroyAllWindows()

# print("finished")